In [1]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math

# list storing types, will be our keys for dicts keeping track of positives and negatives
pokemon_types = ["normal", "fire", "water", "grass", 
                 "electric", "ice", "fighting", "poison",
                 "ground", "flying", "psychic", "bug",
                 "rock", "ghost", "dragon", "dark",
                 "steel", "fairy"]

with open("pokemon_data.json", "r") as file:
    data = json.load(file)
data_df = pd.DataFrame(data)

# shuffling the data set by sampling all of it with a set random seed
data_df = data_df.sample(frac = 1, random_state = 727)
indices = np.array_split(np.arange(len(data_df)), 20)
split_df = [data_df.iloc[idx] for idx in indices]
# splitting 70% of the data into training, 15% each into testing and validation
training_set = pd.concat(split_df[0:14])
testing_set = pd.concat(split_df[14:17])
validation_set = pd.concat(split_df[17:])
# the combined set will be used for testing for algorithms that do not require validation
combined_set = pd.concat(split_df[14:])

In [2]:
# double checking to ensure each set has the right number of pokemon
# can see here the training set has about 700 pokemon, testing and validation set each have about 150
print(training_set.shape)
print(testing_set.shape)
print(validation_set.shape)

(738, 11)
(156, 11)
(156, 11)


In [3]:
# helper function to determine precision, recall, and F1 score for each benchmark
def performance_metrics(true_positives, false_positives, true_negatives, false_negatives):
    precisions = dict.fromkeys(pokemon_types)
    recalls = dict.fromkeys(pokemon_types)
    f1scores = dict.fromkeys(pokemon_types)
    sum_precision = 0
    sum_recall = 0
    sum_f1 = 0

    for pokemon_type in pokemon_types:
        precisions[pokemon_type] = (true_positives[pokemon_type]) / (false_positives[pokemon_type] + true_positives[pokemon_type])
        sum_precision += precisions[pokemon_type]
        recalls[pokemon_type] = (true_positives[pokemon_type]) / (false_negatives[pokemon_type] + true_positives[pokemon_type])
        sum_recall += recalls[pokemon_type]
        f1scores[pokemon_type] = (2 * true_positives[pokemon_type]) / ( (2 * true_positives[pokemon_type]) + false_positives[pokemon_type] + false_negatives[pokemon_type])
        sum_f1 += f1scores[pokemon_type]

    return sum_precision/len(pokemon_types), sum_recall/len(pokemon_types), sum_f1/len(pokemon_types)

In [4]:
# helper function to mutate dicts keeping track of true/false positives/negatives
def mutate_dicts(true_positives, false_positives, true_negatives, false_negatives, actual_types, predicted_type):
    if predicted_type in actual_types:
        for pokemon_type in pokemon_types:
            # if we get the type right,
            # it counts as a true positive for each type of the pokemon,
            # and as a true negative for every other type
            if pokemon_type in actual_types:
                true_positives[pokemon_type] += 1
            else:
                true_negatives[pokemon_type] += 1

    else:
        for pokemon_type in pokemon_types:
            # if we don't get the type right,
            # it counts as a false positive for the predicted type,
            # and a false negative for the types of the pokemon
            if pokemon_type in actual_types:
                false_negatives[pokemon_type] += 1
            elif pokemon_type == predicted_type:
                false_positives[pokemon_type] += 1
    

In [5]:
# majority class is water type
def majority_class_benchmark(testing_set, testing_size):
    true_positives = dict.fromkeys(pokemon_types, 0)
    false_positives = dict.fromkeys(pokemon_types, 0)
    true_negatives = dict.fromkeys(pokemon_types, 0)
    false_negatives = dict.fromkeys(pokemon_types, 0)

    majority_class = "water"
    num_correct = 0
    for index, row in testing_set.iterrows():
        if majority_class in row["types"]:
            num_correct += 1
        mutate_dicts(true_positives, false_positives, true_negatives, false_negatives, row["types"], majority_class)

    for pokemon_type in pokemon_types:
        # this adjustment ensures we don't divide by 0 when we do our precision and recall calculations
        if false_positives[pokemon_type] == 0:
            false_positives[pokemon_type] += 1

    metrics = dict.fromkeys(["accuracy", "precision", "recall", "f1score"])
    metrics["accuracy"] = num_correct/testing_size
    metrics["precision"], metrics["recall"], metrics["f1score"] = performance_metrics(true_positives, false_positives, true_negatives, false_negatives)
    return metrics

In [6]:
# helper function to determine Euclidean distance between two pokemon based on base stats
def distance(pokemon1, pokemon2):
    hp = (pokemon1["hp"] - pokemon2["hp"]) ** 2
    attack = (pokemon1["attack"] - pokemon2["attack"]) ** 2
    defense = (pokemon1["defense"] - pokemon2["defense"]) ** 2
    special_attack = (pokemon1["special-attack"] - pokemon2["special-attack"]) ** 2
    special_defense = (pokemon1["special-defense"] - pokemon2["special-defense"]) ** 2
    speed = (pokemon1["speed"] - pokemon2["speed"]) ** 2
    return math.sqrt(hp + attack + defense + special_attack + special_defense + speed)

In [7]:
def nearest_neighbor_benchmark(training_set, testing_set, testing_size):
    true_positives = dict.fromkeys(pokemon_types, 0)
    false_positives = dict.fromkeys(pokemon_types, 0)
    true_negatives = dict.fromkeys(pokemon_types, 0)
    false_negatives = dict.fromkeys(pokemon_types, 0)

    num_correct = 0
    prediction = ""

    # iterating through each pokemon in our testing set
    for index1, row1 in testing_set.iterrows():
        current_min = 1000000000

        # iterating through each pokemon in our training set
        for index2, row2 in training_set.iterrows():
            current_distance = distance(row1, row2)
            # if a pokemon is our new nearest neighbor, we save its primary type as our prediction
            if current_distance < current_min:
                current_min = current_distance
                prediction = row2["types"][0]

        if prediction in row1["types"]:
            num_correct += 1
        mutate_dicts(true_positives, false_positives, true_negatives, false_negatives, row1["types"], prediction)
        
    metrics = dict.fromkeys(["accuracy", "precision", "recall", "f1score"])
    metrics["accuracy"] = num_correct/testing_size
    metrics["precision"], metrics["recall"], metrics["f1score"] = performance_metrics(true_positives, false_positives, true_negatives, false_negatives)
    return metrics
        

In [8]:
print(f"{majority_class_benchmark(combined_set, combined_set.shape[0])}")

{'accuracy': 0.1858974358974359, 'precision': 0.4992165242165242, 'recall': 0.13173095414300354, 'f1score': 0.14673369554419302}


In [9]:
print(f"{nearest_neighbor_benchmark(training_set, combined_set, combined_set.shape[0])}")

{'accuracy': 0.2467948717948718, 'precision': 0.35407548916098847, 'recall': 0.23879585503344833, 'f1score': 0.2758625822235133}
